In [2]:
# Import required libraries

import nltk
import math
import pandas as pd

from collections import Counter
from nltk.corpus import gutenberg

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# Download required NLTK resources

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')

print("Gutenberg Corpus downloaded successfully.")

Gutenberg Corpus downloaded successfully.


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\mahat\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mahat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mahat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# Load Emma from the Gutenberg Corpus

raw_text = gutenberg.raw('austen-emma.txt')

print("Text loaded successfully.")
print("Number of characters:", len(raw_text))
print("\nFirst 500 characters:\n")
print(raw_text[:500])

Text loaded successfully.
Number of characters: 887071

First 500 characters:

[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her.

She was the youngest of the two daughters of a most affectionate,
indulgent father; and had, in consequence of her sister's marriage,
been mistress of his house from a very early period.  Her mother
had died t


In [5]:
# Tokenize and normalize the corpus

raw_tokens = nltk.word_tokenize(raw_text)

tokens = [
    word.lower()
    for word in raw_tokens
    if word.isalpha()
]

print("Total normalized tokens:", len(tokens))
print("\nFirst 30 normalized tokens:")
print(tokens[:30])

Total normalized tokens: 157114

First 30 normalized tokens:
['emma', 'by', 'jane', 'austen', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessings', 'of']


In [6]:
# Calculate vocabulary

vocabulary = set(tokens)
vocabulary_size = len(vocabulary)

print("Vocabulary size:", vocabulary_size)

Vocabulary size: 6933


In [7]:
# Construct unigram frequency counts

unigram_counts = Counter(tokens)

# Construct bigram frequency counts

bigram_counts = Counter(
    zip(tokens[:-1], tokens[1:])
)

print("Total unigram occurrences:", sum(unigram_counts.values()))
print("Total bigram occurrences:", sum(bigram_counts.values()))

print("\nMost common unigrams:")
print(unigram_counts.most_common(10))

print("\nMost common bigrams:")
print(bigram_counts.most_common(10))

Total unigram occurrences: 157114
Total bigram occurrences: 157113

Most common unigrams:
[('the', 5201), ('to', 5181), ('and', 4877), ('of', 4284), ('i', 3177), ('a', 3124), ('it', 2503), ('her', 2448), ('was', 2396), ('she', 2336)]

Most common bigrams:
[(('to', 'be'), 605), (('of', 'the'), 565), (('in', 'the'), 447), (('it', 'was'), 446), (('i', 'am'), 394), (('she', 'had'), 332), (('she', 'was'), 328), (('had', 'been'), 307), (('it', 'is'), 298), (('i', 'have'), 281)]


In [8]:
# Display the top 10 unigram and bigram frequencies

top_unigrams = pd.DataFrame(
    unigram_counts.most_common(10),
    columns=["Word", "Frequency"]
)

print("Top 10 Unigrams:")
display(top_unigrams)

top_bigrams = pd.DataFrame(
    [
        (" ".join(bigram), frequency)
        for bigram, frequency in bigram_counts.most_common(10)
    ],
    columns=["Bigram", "Frequency"]
)

print("Top 10 Bigrams:")
display(top_bigrams)

Top 10 Unigrams:


,Word,Frequency
0,the,5201
1,to,5181
2,and,4877
3,of,4284
4,i,3177
5,a,3124
6,it,2503
7,her,2448
8,was,2396
9,she,2336


Top 10 Bigrams:


,Bigram,Frequency
0,to be,605
1,of the,565
2,in the,447
3,it was,446
4,i am,394
5,she had,332
6,she was,328
7,had been,307
8,it is,298
9,i have,281


In [9]:
# Add-one smoothed bigram probability

def bigram_probability(word1, word2):
    """
    Calculate the add-one smoothed probability
    P(word2 | word1).
    """

    bigram_count = bigram_counts[(word1, word2)]
    unigram_count = unigram_counts[word1]

    probability = (bigram_count + 1) / (
        unigram_count + vocabulary_size
    )

    return probability

# Test a few bigram probabilities
test_bigrams = [
    ("she", "was"),
    ("the", "man"),
    ("computer", "science")
]

for word1, word2 in test_bigrams:
    probability = bigram_probability(word1, word2)
    print(f"P({word2} | {word1}) = {probability:.8f}")

P(was | she) = 0.03549466
P(man | the) = 0.00115378
P(science | computer) = 0.00014424


In [10]:
# Calculate the log probability of a sentence

def sentence_score(sentence):
    """
    Calculate the log probability of a sentence
    using an add-one smoothed bigram model.
    """

    sentence_tokens = [
        word.lower()
        for word in nltk.word_tokenize(sentence)
        if word.isalpha()
    ]

    if len(sentence_tokens) < 2:
        return 0.0

    log_probability = 0.0

    for word1, word2 in zip(
        sentence_tokens[:-1],
        sentence_tokens[1:]
    ):
        probability = bigram_probability(word1, word2)
        log_probability += math.log(probability)

    return log_probability

In [11]:
# Test the Bigram Language Model on three sentences

test_sentences = [
    "She was happy.",
    "The man walked home.",
    "The computer science student studied."
]

for i, sentence in enumerate(test_sentences, start=1):
    score = sentence_score(sentence)

    print(f"Sentence {i}: {sentence}")
    print(f"Log Probability: {score:.6f}")
    print("-" * 50)

Sentence 1: She was happy.
Log Probability: -11.380644
--------------------------------------------------
Sentence 2: The man walked home.
Log Probability: -23.394720
--------------------------------------------------
Sentence 3: The computer science student studied.
Log Probability: -35.935910
--------------------------------------------------


In [12]:
# Create a table containing sentence scores
            
score_data = []

for sentence in test_sentences:
    score = sentence_score(sentence)
    score_data.append([sentence, score])

score_df = pd.DataFrame(
    score_data,
    columns=["Test Sentence", "Log Probability"]
)

score_df

,Test Sentence,Log Probability
0,She was happy.,-11.380644
1,The man walked home.,-23.394720
2,The computer science student studied.,-35.935910


### Q2(f) Importance of Smoothing

Smoothing is important because many possible word combinations do not
appear in the training corpus. Without smoothing, an unseen bigram would
have a probability of zero.

When probabilities are multiplied to calculate a sentence probability,
a single zero probability would make the entire sentence probability
zero. Add-one smoothing avoids this problem by adding one to every
bigram count.

Therefore, smoothing allows the language model to assign a small
non-zero probability to unseen word combinations and improves its
ability to handle previously unseen sentences.